# Stage 10 - chunking trained on retrieval recall

Design: `docs/stage10_rl_chunking.md`.

## Setup

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/RAG chunk optimize'

from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.isfile(os.path.join(PROJECT_DIR, 'config.py')):
    raise RuntimeError(f'Missing config.py under {PROJECT_DIR!r}.')

os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print('project:', PROJECT_DIR)
print(C.summary())

## Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## NQ train-split cache

In [ ]:
!python scripts/16_build_rerank_train_data.py

## Smoke run

In [ ]:
!python scripts/23_train_rl_chunker.py --smoke

## Train the policy

In [ ]:
!python scripts/23_train_rl_chunker.py --steps 100 --lr 5e-5

## Training curve

In [ ]:
import os, pathlib
import pandas as pd
import matplotlib.pyplot as plt

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
log = pd.read_csv(latest / 'stage10_train_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(log['step'], log['reward_greedy'].rolling(20, min_periods=1).mean())
axes[0].set_xlabel('step'); axes[0].set_ylabel('reward (greedy decode)')
axes[0].set_title('reward, 20-step moving average')
dev = log.dropna(subset=['dev_recall@5'])
axes[1].plot(dev['step'], dev['dev_recall@5'], marker='o')
axes[1].set_xlabel('step'); axes[1].set_ylabel('dev R@5')
axes[1].set_title('held-out dev recall')
fig.tight_layout()
plt.savefig(latest / 'stage10_reward_curve.png', dpi=150)
plt.show()

## Gate

In [ ]:
import csv, os, pathlib
import config as C

latest = pathlib.Path(os.environ['RAG_DATA_ROOT']) / 'results' / 'latest'
k = max(C.RECALL_KS)

def _rows(name):
    with open(latest / name, newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

dev = _rows('stage10_dev_results.csv')
supervised = float(next(r for r in dev if r['method'] == 'transformer')[f'recall@{k}'])
best_rl = max(float(r[f'recall@{k}']) for r in dev if r['method'] == 'rl')
delta = best_rl - supervised

entropy = [float(r['entropy']) for r in _rows('stage10_train_log.csv') if r['entropy']]
# windowed: step-to-step entropy swings ~0.2 nats on batch composition alone
w = max(1, min(10, len(entropy) // 2))
moved = abs(sum(entropy[-w:]) / w - sum(entropy[:w]) / w)

beats_gate = delta >= C.STAGE10_GO_THRESHOLD
policy_moved = moved >= 0.05
GO = beats_gate and policy_moved

print(f'dev R@{k}: best RL {best_rl:.4f} vs supervised {supervised:.4f} '
      f'-> {delta:+.4f}  (gate {C.STAGE10_GO_THRESHOLD:+.3f})')
print(f'policy entropy moved {moved:.4f} nats/decision  (needs >= 0.05)')
if GO:
    print('\nGO - the evaluation cells below will run.')
elif not policy_moved:
    print('\nNO-GO: the policy barely moved, so this is a failed optimisation '
          'rather than a result about the objective.\n'
          'Rerun with --lr 5e-5 --fresh, or extend with a larger --steps. '
          'Do not write this up.')
else:
    print('\nNO-GO: the dev delta is below the gate. That is a legitimate '
          'result - record it as a NO-GO rather than re-rolling the seed.')

## Stage 6 evaluation (GO required)

In [ ]:
if GO:
    !python scripts/24_eval_rl_chunker.py --max-questions 50
else:
    print('skipped: the gate above did not pass.')

In [ ]:
if GO:
    !python scripts/24_eval_rl_chunker.py
else:
    print('skipped: the gate above did not pass.')

## Review

In [ ]:
from IPython.display import Image, Markdown, display

summary = latest / 'stage10_summary.md'
if summary.exists():
    display(Markdown(summary.read_text(encoding='utf-8')))
    display(Image(str(latest / 'stage10_rl_delta.png')))
else:
    print('no Stage 10 summary yet - the evaluation has not run.')

## Archive verified results

In [ ]:
text = summary.read_text(encoding='utf-8') if summary.exists() else ''
check_ok = 'reproduction check vs stage6/final: PASS' in text
invalid = 'Verdict: INVALID' in text

if text and check_ok and not invalid:
    !python scripts/save_stage_results.py --stage stage10
elif not text:
    print('not archived: the evaluation has not produced a summary.')
elif not check_ok:
    print('not archived: the recomputed baselines did not reproduce '
          'stage6/final, so the RL rows are not trustworthy.')
else:
    print('not archived: the verdict is INVALID (chunk size drifted off the '
          'matched baseline).')